In [5]:
import os
import json
import pandas as pd
import requests
import ast
import re
import time
from tqdm import tqdm
from openai import OpenAI
from pypfopt import expected_returns, risk_models, EfficientFrontier
from pypfopt.discrete_allocation import DiscreteAllocation, get_latest_prices

# Configure the API key and URL
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
POLYGON_API_KEY = os.getenv("POLYGON_API_KEY", "")
BASE_URL = (
    "https://api.polygon.io/v2/aggs/ticker/{ticker}/range/"
    "{multiplier}/{timespan}/{from_}/{to}"
)

In [18]:
def generate_portfolios():
    client = OpenAI(api_key=OPENAI_API_KEY)
    prompt = (
        "You are an expert portfolio construction advisor. "
        "Based on the past year of US stock performance, generate a portfolio suggestions "
        "that includes 15-30 TICKERS ONLY based on current market performance"
        "Output must be a pure JSON list, each element containing 'name'."
        "DO NOT INCLUDE ANY EXPLANATIONS, JSON ONLY"
    )
    resp = client.responses.create(
        model="gpt-5", 
        tools=[{"type": "web_search_preview"}], 
        input=prompt
    )
    raw = resp.output_text
    txt_path = os.path.join("D:/my-fin-project/txt_save", "portfolios.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(raw)
    print("Raw portfolio saved to txt_save/portfolios.txt")
    return convert_portfolios_txt_to_json(txt_path, "D:\my-fin-project\json_save\portfolios.json")

# ---------------------------------------------------------------------------------------------------------------------

def convert_portfolios_txt_to_json(input_path: str, output_path: str):
    def _strip_trailing_commas(s: str) -> str:
        s = s.strip()
        s = re.sub(r',(\s*[}\]])', r'\1', s)
        s = s.rstrip(", \t\r\n")
        return s

    with open(input_path, "r", encoding="utf-8-sig") as f:
        raw = f.read().strip()
    if not raw:
        data = []
    else:
        try:
            obj = json.loads(_strip_trailing_commas(raw))
            data = obj if isinstance(obj, list) else [obj]
        except json.JSONDecodeError:
            items = []
            for line in raw.splitlines():
                s = line.strip()
                if not s or s.startswith("//") or s.startswith("#"):
                    continue
                if s.endswith(","):
                    s = s[:-1].rstrip()
                try:
                    items.append(json.loads(_strip_trailing_commas(s)))
                except json.JSONDecodeError:
                    pass
            if items:
                data = items
            else:
                objs = re.findall(r'\{(?:[^{}]|(?R))*\}', raw, flags=re.DOTALL)
                if objs:
                    joined = "[" + ",".join(o.rstrip(", \t\r\n") for o in objs) + "]"
                    try:
                        data = json.loads(_strip_trailing_commas(joined))
                    except json.JSONDecodeError:
                        raise ValueError("Unrecognised file format")
                else:
                    raise ValueError("Unrecognised file format")

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"Parsing complete, saved to {output_path}")
    return data

In [ ]:
generate_portfolios()

In [15]:
def load_portfolios(json_file=None):
    if not json_file:
        json_file = os.path.join("D:/my-fin-project/json_save", "portfolios.json")
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    tickers = [item["name"] for item in data]
    return tickers

In [16]:
def fetch_and_save(tickers, start, end, filename, multiplier=1, timespan="day", sleep_time=1):  
    records = []
    for t in tqdm(tickers, desc="Fetching data"):
        url = BASE_URL.format(ticker=t, multiplier=multiplier, timespan=timespan, from_=start, to=end)
        params = {
            "adjusted": True,
            "sort": "asc",
            "limit": 50000,
            "apiKey": POLYGON_API_KEY
        }
        try:
            resp = requests.get(url, params=params)
            resp.raise_for_status()
            results = resp.json().get("results", [])
            if not results:
                print(f"[warn] {t} has no history, skipping.")
                continue
            for item in results:
                records.append({
                    "date": pd.to_datetime(item["t"], unit="ms"),
                    "ticker": t,
                    "close": item["c"]
                })
        except Exception as e:
            print(f"[error] failed to fetch {t}: {e}")
            continue
        time.sleep(sleep_time)  # rate-limit guard to stay under the API quota

    if not records:
        print("[warn] every ticker came back empty; no CSV written.")
        return None

    # Build the wide format: date index, tickers as columns
    df = pd.DataFrame(records).pivot(index="date", columns="ticker", values="close")
    df.sort_index(inplace=True)
    path = os.path.join("D:/my-fin-project/csv_save", filename)
    df.to_csv(path)
    print(f"[ok] Price data saved to csv_save/{filename}: {df.shape[0]} rows x {df.shape[1]} cols")
    return path

In [ ]:
tickers = load_portfolios()
fetch_and_save(tickers=tickers, start="2024-07-25", end="2025-07-25", filename='test_portfolio.csv')

In [8]:
def analyze_performance(csv_file):
    # Read in price data
    df = pd.read_csv(csv_file, parse_dates=True, index_col="date")
    # Calculate expected returns and sample covariance
    mu = expected_returns.mean_historical_return(df)
    S = risk_models.sample_cov(df)

    # Optimize for maximal Sharpe ratio
    ef = EfficientFrontier(mu, S)
    raw_weights = ef.max_sharpe()
    cleaned_weights = ef.clean_weights()
    ef.save_weights_to_file("weights.csv")  # saves to file
    print(cleaned_weights)
    ef.portfolio_performance(verbose=True)
    # Calculate expected returns and sample covariance
    latest_prices = get_latest_prices(df)
    da = DiscreteAllocation(cleaned_weights, latest_prices, total_portfolio_value=25741.99)
    allocation, leftover = da.greedy_portfolio()
    print("Discrete allocation:", allocation)
    print("Funds remaining: ${:.2f}".format(leftover))


In [ ]:
analyze_performance(r'D:\my-fin-project\csv_save\prices_20250808-192013.csv')

In [22]:
import pandas as pd
import numpy as np
def risk_analysis(csv_file):
    # === Parameters ===
    confidence_level = 0.95
    portfolio_value = 1_000_000  # assume 1,000,000 managed per ticker

    # === Step 1: load data ===
    df = pd.read_csv(csv_file, parse_dates=["date"])
    df = df.set_index("date").sort_index()

    # === Step 2: compute returns for every ticker ===
    returns = df.pct_change().dropna()

    # === Step 3: compute VaR and CVaR in bulk ===
    result = []

    for ticker in returns.columns:
        r = returns[ticker].dropna()
        var = np.percentile(r, (1 - confidence_level) * 100)
        cvar = r[r <= var].mean()
        result.append({
            "Ticker": ticker,
            f"VaR_{int(confidence_level*100)}": var,
            f"CVaR_{int(confidence_level*100)}": cvar,
            "VaR_amount": -var * portfolio_value,
            "CVaR_amount": -cvar * portfolio_value
        })

    # === Step 4: emit the results table ===
    result_df = pd.DataFrame(result)
    result_df[f"VaR_{int(confidence_level*100)}"] = result_df[f"VaR_{int(confidence_level*100)}"].map(lambda x: f"{x:.2%}")
    result_df[f"CVaR_{int(confidence_level*100)}"] = result_df[f"CVaR_{int(confidence_level*100)}"].map(lambda x: f"{x:.2%}")
    print(result_df)


In [ ]:
risk_analysis(r'D:\my-fin-project\csv_save\test_portfolio.csv')

In [ ]:
import os
import requests
from openai import OpenAI

def check_openai():
    key = os.getenv("OPENAI_API_KEY")
    if not key:
        return "[error] OpenAI API key not found in environment variables"
    try:
        client = OpenAI(api_key=key)
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": "Test"}],
            max_tokens=5
        )
        return f"[ok] OpenAI API is working (model: gpt-4o-mini, output: {resp.choices[0].message.content})"
    except Exception as e:
        return f"[error] OpenAI API error: {e}"

def check_polygon():
    key = os.getenv("POLYGON_API_KEY")
    if not key:
        return "[error] Polygon API key not found in environment variables"
    try:
        url = f"https://api.polygon.io/v1/marketstatus/now?apiKey={key}"
        r = requests.get(url, timeout=5)
        if r.status_code == 200:
            return f"[ok] Polygon API is working (status: {r.json().get('market')})"
        else:
            return f"[error] Polygon API HTTP error: {r.status_code} - {r.text}"
    except Exception as e:
        return f"[error] Polygon API error: {e}"

def check_deepseek():
    key = os.getenv("DEEPSEEK_API_KEY")
    if not key:
        return "[error] DeepSeek API key not found in environment variables"
    try:
        client = OpenAI(
            api_key=key,
            base_url="https://api.deepseek.com"  # DeepSeek API endpoint
        )
        resp = client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": "Test"}],
            max_tokens=5
        )
        return f"[ok] DeepSeek API is working (model: deepseek-chat, output: {resp.choices[0].message.content})"
    except Exception as e:
        return f"[error] DeepSeek API error: {e}"

if __name__ == "__main__":
    print(check_openai())
    print(check_polygon())
    print(check_deepseek())


In [ ]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

models = client.models.list()
for m in models.data:
    print(m.id)
